In [1]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = (
    SparkSession.builder
    .appName("MyGoitSparkSandbox")
    .config("spark.python.use.daemon", "false")
    .config("spark.python.worker.reuse", "false")
    .config("spark.sql.shuffle.partitions", "4")
    # Disable Arrow to avoid Python 3.13 serialization crashes
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
    # Increase timeouts so workers aren't killed during shuffle
    .config("spark.network.timeout", "800s")
    .config("spark.executor.heartbeatInterval", "200s")
    .config("spark.python.worker.timeout", "120")
    .getOrCreate()
)

26/04/05 01:16:59 WARN Utils: Your hostname, Kalnysh.local resolves to a loopback address: 127.0.0.1; using 192.168.2.33 instead (on interface en0)
26/04/05 01:16:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/05 01:17:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import pandas as pd
import requests
import tempfile
import os
from io import StringIO

url = "https://data.sfgov.org/resource/nuek-vuh3.csv?$limit=50000"

headers = {"Accept": "application/json"}
response = requests.get(url, headers=headers)
response.raise_for_status()

pandas_df = pd.read_csv(StringIO(response.text), low_memory=False)
pandas_df["box"] = pandas_df["box"].astype(str)

# Write to parquet and read back via Spark's native JVM reader.
# spark.createDataFrame(pandas_df) embeds data as a LocalRelation that requires
# Python workers for every action — those workers crash on this environment.
# Reading from parquet uses no Python workers at all.
tmpdir = tempfile.mkdtemp()
tmpfile = os.path.join(tmpdir, "nuek.parquet")
pandas_df.to_parquet(tmpfile, index=False)
nuek_df = spark.read.parquet(tmpfile)

print(f"Loaded {nuek_df.count()} rows, {len(nuek_df.columns)} columns")
nuek_df.printSchema()

Loaded 50000 rows, 37 columns
root
 |-- call_number: long (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- incident_number: long (nullable = true)
 |-- call_type: string (nullable = true)
 |-- call_date: string (nullable = true)
 |-- watch_date: string (nullable = true)
 |-- received_dttm: string (nullable = true)
 |-- entry_dttm: string (nullable = true)
 |-- dispatch_dttm: string (nullable = true)
 |-- response_dttm: string (nullable = true)
 |-- on_scene_dttm: string (nullable = true)
 |-- transport_dttm: string (nullable = true)
 |-- hospital_dttm: string (nullable = true)
 |-- call_final_disposition: string (nullable = true)
 |-- available_dttm: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zipcode_of_incident: double (nullable = true)
 |-- battalion: string (nullable = true)
 |-- station_area: double (nullable = true)
 |-- box: string (nullable = true)
 |-- original_priority: string (nullable = true)
 |-- pri

In [3]:
# Створюємо тимчасове представлення для виконання SQL-запитів
nuek_df.createOrReplaceTempView("nuek_view")

26/04/05 01:17:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
# Виконуємо SQL-маніпуляції
spark.sql("SELECT * FROM nuek_view LIMIT 10").show()

+-----------+-------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+-------------+-------------------+---------+------------+----+-----------------+--------+--------------+--------+--------------------+----------------+--------------+------------------------------+------------------------+-------------------+---------------------------------+--------------+--------------------+--------------------+--------------------+---------------------------+
|call_number|unit_id|incident_number|           call_type|           call_date|          watch_date|       received_dttm|          entry_dttm|       dispatch_dttm|       response_dttm|       on_scene_dttm|      transport_dttm|       hospital_dttm|call_final_disposition|      available_dttm|             a

In [5]:
from pyspark.sql.functions import col

# Скільки унікальних call_type є в датасеті?
res = nuek_df.select('call_type')\
      .where(col("call_type").isNotNull())\
      .distinct()\
      .count()

print(res)

23


In [6]:
# Скільки унікальних call_type є в датасеті? (з використанням SQL)
df = spark.sql("""SELECT COUNT(DISTINCT call_type) as count
                    FROM nuek_view
                    WHERE call_type IS NOT NULL""")
# Виводимо датафрейм на дисплей
df.show()

+-----+
|count|
+-----+
|   23|
+-----+



In [7]:
# Витягуємо дані колонки з датафрейму
print(df.collect(), type(df.collect()))
# Дотягуємось до самого значення за номером рядка та іменем колонки
print(df.collect()[0]['count'])
# або за номером рядка та номером колонки
print(df.collect()[0][0])

[Row(count=23)] <class 'list'>
23
23


In [8]:
# Які call_type є найбільш популярними (топ-3)?
nuek_df.groupBy('call_type') \
    .count() \
    .orderBy(col('count').desc()) \
    .limit(3) \
    .show()

# Які call_type є найбільш популярними (топ-3)? (з використанням SQL)
spark.sql("""
    SELECT call_type, COUNT(*) AS count
    FROM nuek_view
    WHERE call_type IS NOT NULL
    GROUP BY call_type
    ORDER BY count DESC
    LIMIT 3
""").show()

+--------------------+-----+
|           call_type|count|
+--------------------+-----+
|    Medical Incident|34846|
|Structure Fire / ...| 4817|
|              Alarms| 4812|
+--------------------+-----+

+--------------------+-----+
|           call_type|count|
+--------------------+-----+
|    Medical Incident|34846|
|Structure Fire / ...| 4817|
|              Alarms| 4812|
+--------------------+-----+



In [9]:
nuek_df.select("received_dttm", "response_dttm") \
    .withColumn("delay_s", col("response_dttm") - (col("received_dttm"))) \
    .show(15)

+--------------------+--------------------+-------+
|       received_dttm|       response_dttm|delay_s|
+--------------------+--------------------+-------+
|2016-04-03T23:15:...|2016-04-03T23:18:...|   NULL|
|2016-04-11T13:14:...|2016-04-11T13:21:...|   NULL|
|2016-04-02T07:48:...|2016-04-02T07:50:...|   NULL|
|2016-04-02T13:08:...|2016-04-02T13:13:...|   NULL|
|2016-04-01T13:20:...|2016-04-01T13:23:...|   NULL|
|2016-04-07T17:35:...|2016-04-07T17:38:...|   NULL|
|2016-04-08T10:28:...|2016-04-08T10:38:...|   NULL|
|2016-04-09T23:07:...|2016-04-09T23:08:...|   NULL|
|2016-04-08T02:13:...|2016-04-08T02:24:...|   NULL|
|2016-04-13T08:52:...|2016-04-13T08:54:...|   NULL|
|2016-04-04T01:12:...|2016-04-04T01:13:...|   NULL|
|2016-04-10T19:29:...|2016-04-10T19:31:...|   NULL|
|2016-04-05T04:03:...|2016-04-05T04:09:...|   NULL|
|2016-04-11T11:43:...|2016-04-11T11:43:...|   NULL|
|2016-04-07T18:29:...|2016-04-07T18:31:...|   NULL|
+--------------------+--------------------+-------+
only showing

In [10]:
from pyspark.sql.types import TimestampType

# Повторюємо рахування колонок, тільки попередньо
# перетворюємо колонки в тип Timestamp
df_times = nuek_df.select("received_dttm", "response_dttm") \
    .withColumn("received_dttm", col("received_dttm").cast(TimestampType())) \
    .withColumn("response_dttm", col("response_dttm").cast(TimestampType())) \
    .withColumn("delay_s", col("response_dttm") - (col("received_dttm")))

# Перевіряємо схему нової таблиці
df_times.printSchema()
# Дивимось на результат
df_times.show(5)


root
 |-- received_dttm: timestamp (nullable = true)
 |-- response_dttm: timestamp (nullable = true)
 |-- delay_s: interval day to second (nullable = true)

+-------------------+-------------------+--------------------+
|      received_dttm|      response_dttm|             delay_s|
+-------------------+-------------------+--------------------+
|2016-04-03 23:15:12|2016-04-03 23:18:45|INTERVAL '0 00:03...|
|2016-04-11 13:14:47|2016-04-11 13:21:06|INTERVAL '0 00:06...|
|2016-04-02 07:48:03|2016-04-02 07:50:48|INTERVAL '0 00:02...|
|2016-04-02 13:08:02|2016-04-02 13:13:09|INTERVAL '0 00:05...|
|2016-04-01 13:20:24|2016-04-01 13:23:44|INTERVAL '0 00:03...|
+-------------------+-------------------+--------------------+
only showing top 5 rows



In [11]:
from pyspark.sql.functions import unix_timestamp

df_times = nuek_df.select("received_dttm", "response_dttm") \
    .withColumn("received_dttm", col("received_dttm").cast(TimestampType())) \
    .withColumn("response_dttm", col("response_dttm").cast(TimestampType())) \
    .withColumn("delay_s", unix_timestamp(col("response_dttm")) - unix_timestamp(col("received_dttm")))

df_times.printSchema()
df_times.show(5)


root
 |-- received_dttm: timestamp (nullable = true)
 |-- response_dttm: timestamp (nullable = true)
 |-- delay_s: long (nullable = true)

+-------------------+-------------------+-------+
|      received_dttm|      response_dttm|delay_s|
+-------------------+-------------------+-------+
|2016-04-03 23:15:12|2016-04-03 23:18:45|    213|
|2016-04-11 13:14:47|2016-04-11 13:21:06|    379|
|2016-04-02 07:48:03|2016-04-02 07:50:48|    165|
|2016-04-02 13:08:02|2016-04-02 13:13:09|    307|
|2016-04-01 13:20:24|2016-04-01 13:23:44|    200|
+-------------------+-------------------+-------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import count, count_if, avg, when, min, max

df_times.groupby().agg(
    count("*").alias("total_"),
    count_if(col("delay_s").isNotNull()).alias("delayed_not_null"),
    count_if(col("delay_s").isNull()).alias("delayed_null"),
    min(col("delay_s")).alias("min_delay"),
    max(col("delay_s")).alias("max_delay"),
    avg(
        col("delay_s")
    ).alias("avg_delay"),
    avg(
        when(col("delay_s").isNotNull(), col("delay_s")).otherwise(0)
    ).alias("avg_zeroed"),
    avg(
        when(col("delay_s").isNotNull(), col("delay_s")).otherwise(220.0615)
    ).alias("avg_replaced")
).show(10)


+------+----------------+------------+---------+---------+-----------------+----------+------------------+
|total_|delayed_not_null|delayed_null|min_delay|max_delay|        avg_delay|avg_zeroed|      avg_replaced|
+------+----------------+------------+---------+---------+-----------------+----------+------------------+
| 50000|           48531|        1469|        0|    19897|230.8675897879706|  224.0847|230.55010686999614|
+------+----------------+------------+---------+---------+-----------------+----------+------------------+



In [13]:
from pyspark.sql.functions import collect_list, array_union, array_distinct

zip_station = nuek_df.select('zipcode_of_incident', 'station_area') \
    .withColumnRenamed("station_area", "station_area_1")

nuek_df.join(zip_station, nuek_df.zipcode_of_incident == zip_station.zipcode_of_incident, 'inner') \
      .drop(zip_station.zipcode_of_incident) \
      .select('zipcode_of_incident', 'station_area', 'station_area_1') \
      .dropDuplicates(['station_area', 'station_area_1']) \
      .dropna() \
      .where(col('station_area') != col('station_area_1')) \
      .groupBy('zipcode_of_incident') \
      .agg(
          collect_list("station_area").alias("station_area_list"),
          collect_list("station_area_1").alias("station_area_list_1")
          ) \
      .withColumn("station_area_united", array_union('station_area_list', 'station_area_list_1')) \
      .withColumn("station_area_distinct", array_distinct('station_area_united')) \
      .show()


+-------------------+--------------------+--------------------+--------------------+---------------------+
|zipcode_of_incident|   station_area_list| station_area_list_1| station_area_united|station_area_distinct|
+-------------------+--------------------+--------------------+--------------------+---------------------+
|            94109.0|[3.0, 3.0, 3.0, 3...|[41.0, 16.0, 28.0...|[3.0, 38.0, 28.0,...| [3.0, 38.0, 28.0,...|
|            94103.0|[6.0, 1.0, 1.0, 1...|[36.0, 8.0, 7.0, ...|[6.0, 1.0, 8.0, 3...| [6.0, 1.0, 8.0, 3...|
|            94107.0|[8.0, 8.0, 8.0, 8...|[29.0, 37.0, 25.0...|[8.0, 1.0, 4.0, 3...| [8.0, 1.0, 4.0, 3...|
|            94110.0|[9.0, 9.0, 9.0, 7...|[6.0, 3.0, 44.0, ...|[9.0, 7.0, 37.0, ...| [9.0, 7.0, 37.0, ...|
|            94105.0|[35.0, 35.0, 13.0...|[48.0, 1.0, 8.0, ...|[35.0, 13.0, 48.0...| [35.0, 13.0, 48.0...|
|            94117.0|[21.0, 21.0, 21.0...|[12.0, 5.0, 22.0,...|[21.0, 12.0, 6.0,...| [21.0, 12.0, 6.0,...|
|            94132.0|[19.0, 19.0, 33.

In [15]:
spark.stop()